# Many-Facet Partial Credit Model (MFPCM) --- Bayesian Estimation with Stan

## 학습 목표

1. MFPCM과 MFRM의 차이를 설명할 수 있다.
2. 문항별 고유 단계 파라미터($\tau_{ik}$)의 의미를 해석할 수 있다.
3. Stan으로 MFPCM을 추정하고 결과를 시각화할 수 있다.

---

## 1. MFPCM 모델이란? (Many-Facet Partial Credit Model)

**MFPCM**은 MFRM의 공통 단계 파라미터($\tau_k$)를 **문항별로 다른** $\tau_{ik}$로 교체한 모델입니다.
`FACETS` 소프트웨어의 `Models= ?,?B,?,R#` 설정이 이 모델에 해당합니다.

$$\ln \left( \frac{P_{nijk}}{P_{nij(k-1)}} \right) = \theta_n - \beta_i - \rho_j - \tau_{ik}$$

| 모델 | 단계 파라미터 | 특징 |
|---|---|---|
| MFRM  | $\tau_k$ (공통) | 모든 문항이 동일한 단계 구조 |
| MFPCM | $\tau_{ik}$ (문항별) | 각 문항이 고유한 단계 구조 |

---

## 2. 식별 가능성

```stan
// Item-specific tau with sum-to-zero per item:
for (i in 1:I)
  tau[i] = append_col(tau_free[i], -sum(tau_free[i]));
// sum(tau[i]) = 0 for each item
```

이 제약으로 $\beta_i$는 문항의 **평균 난이도**를 나타내고,
$\tau_{ik}$는 각 단계가 그 평균으로부터 얼마나 벗어나는지를 표현합니다.

---

## 3. MFRM vs MFPCM: 어느 모델이 맞는가?

**MFRM 가정**: 모든 문항이 동일한 척도 구조를 공유함 (Likert 항목들이 동일한 의미)  
**MFPCM 가정**: 각 문항이 서로 다른 단계 구조를 가질 수 있음  
→ **한국어 평가**에서 쓰기, 말하기 루브릭의 각 항목(내용, 구조, 언어)이 서로 다른 단계 구조를 가질 때 MFPCM이 더 적합


In [ ]:
import sys, os, warnings, tempfile
import matplotlib as mpl, matplotlib.font_manager as fm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmdstanpy
from scipy import stats

warnings.filterwarnings('ignore')

def setup_korean_font():
    candidates = {
        'win32':  [('C:/Windows/Fonts/malgun.ttf', 'Malgun Gothic')],
        'darwin': [('/System/Library/Fonts/AppleSDGothicNeo.ttc', 'Apple SD Gothic Neo')],
        'linux':  [('/usr/share/fonts/truetype/nanum/NanumGothic.ttf', 'NanumGothic')],
    }
    for path, name in candidates.get(sys.platform, []):
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            mpl.rcParams['font.family'] = [name, 'DejaVu Sans']
            return
    mpl.rcParams['font.family'] = ['DejaVu Sans']

setup_korean_font()
mpl.rcParams['axes.unicode_minus'] = False
np.random.seed(5101)


## 4. 시뮬레이션 설계

- **N=100** 피험자, **I=4** 문항, **R=5** 채점자, **K=3** 카테고리
- **문항별 고유 단계 파라미터** ($\tau_{ik}$):
  - 문항 1: 이상적 순서형 (ordered) → [-1.0, 1.0]
  - 문항 2: 역순형 (disordered) → [0.8, -0.8]
  - 문항 3, 4: 중간 형태


In [ ]:
N = 100; I = 4; R = 5; K = 3

theta_true = np.random.normal(0, 1, N)

beta_raw  = np.array([-0.8, -0.2, 0.4, 1.0])
beta_true = beta_raw - beta_raw.mean()

rho_raw   = np.array([-0.8, -0.3, 0.1, 0.4, 0.9])
rho_true  = rho_raw - rho_raw.mean()

# Item-specific tau (K-1 = 2 step params per item, sum-to-zero)
# Item 1: well-ordered (prototype of good item design)
# Item 2: disordered (bad design)
tau_true = np.array([
    [-1.0,  1.0],   # Item 1: ordered, Category 1 peaks in the middle
    [ 0.8, -0.8],   # Item 2: disordered - no modal middle category
    [-0.3,  0.3],   # Item 3: mild ordering
    [ 0.5, -0.5],   # Item 4: reversed
])
# Enforce sum-to-zero for each item
for i in range(I):
    tau_true[i] -= tau_true[i].mean()

print("True parameters:")
print(f"  beta:  {np.round(beta_true, 3)}")
print(f"  rho:   {np.round(rho_true, 3)}")
print("  tau (item-specific):")
for i in range(I):
    ordered = 'ORDERED' if tau_true[i, 0] < tau_true[i, 1] else 'disordered'
    print(f"    Item {i+1}: {np.round(tau_true[i], 3)}  ({ordered})")


In [ ]:
def mfpcm_probs(theta, beta_i, rho_j, tau_i):
    log_p = np.zeros(K)
    for k in range(1, K):
        log_p[k] = log_p[k-1] + (theta - beta_i - rho_j - tau_i[k-1])
    log_p -= log_p.max()
    probs = np.exp(log_p)
    return probs / probs.sum()

Y_nir = np.zeros((N, I, R), dtype=int)
for n in range(N):
    for i in range(I):
        for r in range(R):
            pr = mfpcm_probs(theta_true[n], beta_true[i], rho_true[r], tau_true[i])
            Y_nir[n, i, r] = np.random.choice(K, p=pr)

print(f"Y_nir shape: {Y_nir.shape}")
print(f"Category counts: {np.bincount(Y_nir.ravel())}")

# Compare category distributions per item
fig, axes = plt.subplots(1, I, figsize=(13, 4), sharey=True)
for i in range(I):
    item_scores = Y_nir[:, i, :].ravel()
    counts = np.bincount(item_scores, minlength=K)
    axes[i].bar(range(K), counts / counts.sum(), color=['#3498DB','#E67E22','#2ECC71'],
                edgecolor='white', alpha=0.85)
    ordered = 'Ordered' if tau_true[i, 0] < tau_true[i, 1] else 'Disordered'
    axes[i].set_title(f'Item {i+1}\n({ordered}\nbeta={beta_true[i]:.2f})', fontsize=10)
    axes[i].set_xticks(range(K))
    axes[i].set_xticklabels([f'Score {k}' for k in range(K)], fontsize=8)
    if i == 0: axes[i].set_ylabel('Proportion', fontsize=11)
axes[0].set_ylim(0, 0.7)
fig.suptitle('Item Response Distributions (MFPCM Simulation)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Stan Model (MFPCM)

MFRM과의 핵심 차이:
- MFRM: `tau[k-1]` (공통 단계 파라미터)
- MFPCM: `tau[ii[obs], k-1]` (문항별 단계 파라미터)


In [ ]:
stan_code = 'data {\n  int<lower=1> N;\n  int<lower=1> I;\n  int<lower=1> R;\n  int<lower=2> K;\n  int<lower=0> Nobs;\n  array[Nobs] int<lower=1,upper=N> nn;\n  array[Nobs] int<lower=1,upper=I> ii;\n  array[Nobs] int<lower=1,upper=R> rr;\n  array[Nobs] int<lower=1,upper=K> y;\n}\nparameters {\n  vector[N] theta;\n  real<lower=0> sigma_theta;\n  vector[I-1] beta_free;\n  vector[R-1] rho_free;\n  matrix[I, K-2] tau_free;   // item-specific step difficulties\n}\ntransformed parameters {\n  vector[I] beta;\n  vector[R] rho;\n  matrix[I, K-1] tau;\n  beta = append_row(beta_free, -sum(beta_free));\n  rho  = append_row(rho_free,  -sum(rho_free));\n  for (i in 1:I)\n    tau[i] = append_col(tau_free[i], -sum(tau_free[i]));\n}\nmodel {\n  theta       ~ normal(0, sigma_theta);\n  sigma_theta ~ exponential(1);\n  beta_free   ~ normal(0, 2);\n  rho_free    ~ normal(0, 1);\n  to_vector(tau_free) ~ normal(0, 2);\n\n  for (obs in 1:Nobs) {\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[nn[obs]] - beta[ii[obs]] - rho[rr[obs]] - tau[ii[obs], k-1]);\n    y[obs] ~ categorical_logit(log_p);\n  }\n}\ngenerated quantities {\n  array[Nobs] int y_rep;\n  for (obs in 1:Nobs) {\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[nn[obs]] - beta[ii[obs]] - rho[rr[obs]] - tau[ii[obs], k-1]);\n    y_rep[obs] = categorical_logit_rng(log_p);\n  }\n}'

nn_arr, ii_arr, rr_arr, y_arr = [], [], [], []
for n in range(N):
    for i in range(I):
        for r in range(R):
            nn_arr.append(n + 1)
            ii_arr.append(i + 1)
            rr_arr.append(r + 1)
            y_arr.append(int(Y_nir[n, i, r]) + 1)

stan_data = {
    'N': N, 'I': I, 'R': R, 'K': K,
    'Nobs': N * I * R,
    'nn': nn_arr, 'ii': ii_arr, 'rr': rr_arr, 'y': y_arr
}

tmpdir = tempfile.mkdtemp()
stan_path = os.path.join(tmpdir, 'mfpcm.stan')
with open(stan_path, 'w') as f:
    f.write(stan_code)

model = cmdstanpy.CmdStanModel(stan_file=stan_path)
print('MFPCM Stan model compiled.')


## 6a. MAP Estimation

In [ ]:
fit_map = model.optimize(data=stan_data, seed=5101, jacobian=True)
theta_map = fit_map.stan_variable('theta')
beta_map  = fit_map.stan_variable('beta')
rho_map   = fit_map.stan_variable('rho')
tau_map   = fit_map.stan_variable('tau')   # shape (I, K-1)

z96 = stats.norm.ppf(0.98)
se_theta = 1.0 / np.sqrt(I * R * 0.25)
se_beta  = 1.0 / np.sqrt(N * R * 0.25)
se_rho   = 1.0 / np.sqrt(N * I * 0.25)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
psets = [(theta_true, theta_map, se_theta, 'steelblue', 'theta (person ability)'),
         (beta_true,  beta_map,  se_beta,  'coral',     'beta (item difficulty)'),
         (rho_true,   rho_map,   se_rho,   'forestgreen','rho (rater severity)')]

for ax, (true, est, se, col, title) in zip(axes, psets):
    for k in range(len(true)):
        ax.plot([true[k]]*2, [est[k]-z96*se, est[k]+z96*se],
                color=col, alpha=0.4, linewidth=1.2)
    ax.scatter(true, est, color=col, s=50, zorder=5, alpha=0.9)
    lo = min(true.min(), est.min()) - 0.3
    hi = max(true.max(), est.max()) + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2)
    r = np.corrcoef(true, est)[0, 1]
    ax.set_xlabel('True', fontsize=10); ax.set_ylabel('MAP', fontsize=10)
    ax.set_title(f'{title}\nr={r:.3f}', fontsize=11)

fig.suptitle('MFPCM MAP Estimation (Laplace 96% CI)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Report tau recovery
print("\nTau (step difficulty) MAP recovery:")
for i in range(I):
    print(f"  Item {i+1}: true={np.round(tau_true[i], 3)}, MAP={np.round(tau_map[i], 3)}")


## 6b. Full Bayesian MCMC

In [ ]:
fit = model.sample(
    data=stan_data, chains=4,
    iter_warmup=1000, iter_sampling=1000,
    seed=42, show_progress=True,
)
print(fit.diagnose())


## 7. MCMC Convergence Analysis

In [ ]:
summary = fit.summary()
theta_s = summary[summary.index.str.startswith('theta[')]
beta_s  = summary[summary.index.str.startswith('beta[')]
rho_s   = summary[summary.index.str.startswith('rho[')]
tau_s   = summary[summary.index.str.startswith('tau[')]

print("Convergence summary:")
for name, df in [('theta', theta_s), ('beta', beta_s), ('rho', rho_s), ('tau', tau_s)]:
    ess_col = 'N_Eff' if 'N_Eff' in df.columns else 'ESS_bulk'
    print(f"  {name:5s}: Rhat max={df['R_hat'].max():.4f}, ESS min={df[ess_col].min():.0f}")

theta_samples = fit.stan_variable('theta')
beta_samples  = fit.stan_variable('beta')
rho_samples   = fit.stan_variable('rho')
tau_samples   = fit.stan_variable('tau')   # (4000, I, K-1)

# Rhat distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
all_rhats = np.concatenate([theta_s['R_hat'].values, beta_s['R_hat'].values,
                             rho_s['R_hat'].values, tau_s['R_hat'].values])
axes[0].hist(all_rhats, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(1.01, color='red', linestyle='--', linewidth=1.8, label='1.01')
axes[0].set_xlabel('R-hat'); axes[0].set_title('All Parameters: R-hat Distribution')
axes[0].legend()

# Tau posteriors
tau_pm = tau_samples.mean(axis=0)   # (I, K-1)
x_pos = np.arange(I)
for k in range(K-1):
    axes[1].errorbar(x_pos + k*0.1, tau_pm[:, k],
                     yerr=[tau_pm[:, k] - np.percentile(tau_samples[:, :, k], 2, axis=0),
                           np.percentile(tau_samples[:, :, k], 98, axis=0) - tau_pm[:, k]],
                     fmt='o-', capsize=4, label=f'tau_{k+1}')
    axes[1].scatter(x_pos + k*0.1, tau_true[:, k], marker='^', s=80, zorder=10)
axes[1].set_xticks(range(I))
axes[1].set_xticklabels([f'Item {i+1}' for i in range(I)])
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('tau value (logit)')
axes[1].set_title('Item-specific tau: Posterior mean (96% CI)\nTriangle = true value')
axes[1].legend(fontsize=9)

plt.suptitle('MFPCM Convergence and Tau Recovery', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 8. Posterior Predictive Check (PPC)

In [ ]:
y_rep_all = fit.stan_variable('y_rep')
y_flat = np.array(y_arr) - 1
obs_mean = y_flat.mean()
rep_means = (y_rep_all - 1).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(rep_means, bins=40, density=True, color='steelblue', alpha=0.7,
             edgecolor='white', label='y_rep distribution')
axes[0].axvline(obs_mean, color='red', linewidth=2.5, label=f'Observed={obs_mean:.3f}')
axes[0].set_xlabel('Mean score'); axes[0].set_title('PPC: Mean Score')
axes[0].legend()

y_mat = np.array(y_arr).reshape(N, I, R) - 1
obs_item = y_mat.mean(axis=(0, 2))
rep_mat  = (y_rep_all - 1).reshape(-1, N, I, R)
rep_item = rep_mat.mean(axis=(1, 3))
axes[1].fill_between(range(I), np.percentile(rep_item, 2, axis=0),
                     np.percentile(rep_item, 98, axis=0), alpha=0.3, color='steelblue', label='96% PI')
axes[1].plot(range(I), rep_item.mean(axis=0), 'b-o', markersize=6, label='Predicted mean')
axes[1].plot(range(I), obs_item, 'r-s', markersize=8, label='Observed')
axes[1].set_xticks(range(I)); axes[1].set_xticklabels([f'Item {i+1}' for i in range(I)])
axes[1].set_ylabel('Mean Score'); axes[1].set_title('PPC: Item Mean Scores (96% PI)')
axes[1].legend()

fig.suptitle('PPC: MFPCM', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
bp = (rep_means >= obs_mean).mean()
print(f"Bayesian p-value: {bp:.3f}")


## 9. Scatter Plot: True vs Posterior Mean

In [ ]:
theta_pm = theta_samples.mean(axis=0)
beta_pm  = beta_samples.mean(axis=0)
rho_pm   = rho_samples.mean(axis=0)
tau_pm   = tau_samples.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (samps, true, pm, col, title) in zip(axes, [
    (theta_samples, theta_true, theta_pm, 'steelblue', 'Person Ability theta'),
    (beta_samples,  beta_true,  beta_pm,  'coral',     'Item Difficulty beta'),
    (rho_samples,   rho_true,   rho_pm,   'forestgreen','Rater Severity rho'),
]):
    ci_lo = np.percentile(samps, 2, axis=0)
    ci_hi = np.percentile(samps, 98, axis=0)
    xs = true[np.argsort(true)]
    ax.fill_between(xs, ci_lo[np.argsort(true)], ci_hi[np.argsort(true)],
                    alpha=0.2, color=col, label='96% CI')
    ax.scatter(true, pm, color=col, s=55, zorder=5, alpha=0.9)
    lo = min(true.min(), pm.min()) - 0.3
    hi = max(true.max(), pm.max()) + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2, label='y=x')
    r = np.corrcoef(true, pm)[0, 1]
    rmse = np.sqrt(np.mean((true - pm)**2))
    ax.set_xlabel('True value'); ax.set_ylabel('Posterior mean')
    ax.set_title(f'{title}\nr={r:.3f}, RMSE={rmse:.3f}', fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('MFPCM: True vs Posterior Mean (96% CI bands)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 10. Probability Curves per Item (문항별 확률 곡선)

MFPCM에서 문항별 카테고리 응답 확률을 비교합니다.
**문항 1** (순서형 tau)과 **문항 2** (비순서형 tau)의 차이를 확인하세요.


In [ ]:
theta_range = np.linspace(-4, 4, 300)
cat_colors = ['#3498DB', '#E67E22', '#2ECC71']
cat_labels = [f'Score {k}' for k in range(K)]

fig, axes = plt.subplots(1, I, figsize=(14, 5), sharey=True)
for i in range(I):
    ax = axes[i]
    b = beta_pm[i]
    tau_i = tau_pm[i]

    probs = np.zeros((len(theta_range), K))
    for t_idx, t in enumerate(theta_range):
        log_p = np.zeros(K)
        for k in range(1, K):
            log_p[k] = log_p[k-1] + (t - b - 0 - tau_i[k-1])  # rho=0 (average rater)
        log_p -= log_p.max()
        p = np.exp(log_p)
        probs[t_idx] = p / p.sum()

    for k in range(K):
        ax.plot(theta_range, probs[:, k], color=cat_colors[k], linewidth=2,
                label=cat_labels[k] if i==0 else '')
        peak_idx = np.argmax(probs[:, k])
        ax.axvline(theta_range[peak_idx], color=cat_colors[k], linestyle=':', alpha=0.35)

    # item mean difficulty as representative
    item_rep_diff = b  # beta_i = item center
    ax.axvline(item_rep_diff, color='black', linestyle='--', linewidth=1.5, alpha=0.8)

    ordered_str = 'ORDERED' if tau_pm[i, 0] < tau_pm[i, 1] else 'disordered'
    ax.set_title(f'Item {i+1} ({ordered_str})\nbeta={b:.2f}, tau=[{tau_i[0]:.2f},{tau_i[1]:.2f}]', fontsize=10)
    ax.set_xlabel('Ability theta', fontsize=10)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)

axes[0].set_ylabel('Probability', fontsize=11)
axes[0].legend(fontsize=8, loc='upper left')
fig.suptitle('MFPCM: Category Response Probability Curves per Item\n(rho=0; dashed = item mean difficulty)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 11. Wright Map (MFPCM)

In [ ]:
fig = plt.figure(figsize=(13, 9))
gs  = gridspec.GridSpec(1, 3, width_ratios=[2.5, 1.2, 0.9], wspace=0.08)
ax_p = fig.add_subplot(gs[0]); ax_i = fig.add_subplot(gs[1]); ax_r = fig.add_subplot(gs[2])
y_range = (-4, 4)

ax_p.hist(theta_pm, bins=16, orientation='horizontal', color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_range); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency'); ax_p.set_ylabel('Logit Scale')
ax_p.set_title('Person Abilities'); ax_p.axhline(0, color='gray', linestyle='--', alpha=0.5)

step_colors = ['#2196F3', '#FF5722']
for i in range(I):
    b = beta_pm[i]
    ax_i.plot([0.1, 0.45], [b, b], color='coral', linewidth=3)
    ax_i.text(0.5, b, f'I{i+1} (b={b:.2f})', fontsize=8, va='center', color='darkred')
    for k in range(K-1):
        step_loc = b + tau_pm[i, k]
        ax_i.plot([0.55, 0.8], [step_loc, step_loc], color=step_colors[k], linewidth=1.5, alpha=0.7)
        ax_i.text(0.82, step_loc, f't{k+1}', fontsize=7, va='center', color=step_colors[k])

ax_i.set_ylim(y_range); ax_i.set_xlim(0, 1.2); ax_i.set_xticks([])
ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('Items + Steps (item-specific tau)'); ax_i.axhline(0, color='gray', linestyle='--', alpha=0.5)

for r in range(R):
    rv = rho_pm[r]
    ax_r.plot([0.1, 0.4], [rv, rv], color='forestgreen', linewidth=2.5)
    ax_r.text(0.45, rv, f'R{r+1}', fontsize=8, va='center', color='darkgreen')
ax_r.set_ylim(y_range); ax_r.set_xlim(0, 0.8); ax_r.set_xticks([])
ax_r.yaxis.set_label_position('right'); ax_r.yaxis.tick_right()
ax_r.set_title('Rater Severity'); ax_r.axhline(0, color='gray', linestyle='--', alpha=0.5)

fig.suptitle('Wright Map --- MFPCM (item-specific step parameters)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 12. Uncertainty Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (samps, true, col, label) in zip(axes, [
    (theta_samples, theta_true, 'steelblue', 'theta'),
    (beta_samples,  beta_true,  'coral',     'beta'),
    (rho_samples,   rho_true,   'forestgreen','rho'),
]):
    ci_w = np.percentile(samps, 98, axis=0) - np.percentile(samps, 2, axis=0)
    sort_idx = np.argsort(true)
    ax.bar(range(len(true)), ci_w[sort_idx], color=col, alpha=0.75, edgecolor='white')
    ax.axhline(ci_w.mean(), color='red', linestyle='--', linewidth=1.8, label=f'Mean={ci_w.mean():.3f}')
    ax.set_xlabel(f'{label} (sorted)'); ax.set_ylabel('96% CI Width'); ax.set_title(f'{label}: CI Width')
    ax.legend(fontsize=8)

fig.suptitle('MFPCM Uncertainty Analysis: 96% Posterior CI Widths', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 13. References / 참고 문헌### 국제 학술지 (확인된 논문)1. **Uto, M., & Ueno, M. (2020)**. A generalized many-facet Rasch model and its Bayesian estimation using Hamiltonian Monte Carlo. *Behaviormetrika, 47*, 469–496. https://doi.org/10.1007/s41237-020-00115-72. **Uto, M. (2022)**. A multidimensional generalized many-facet Rasch model for rubric-based performance assessment. *Behaviormetrika*. https://doi.org/10.1007/s41237-021-00144-w3. **Masters, G. N. (1982)**. A Rasch model for partial credit scoring. *Psychometrika, 47*(2), 149–174. *(PCM 원전)*4. **Bürkner, P.-C. (2021)**. Bayesian Item Response Modeling in R with brms and Stan. *Journal of Statistical Software, 100*(5), 1–54. https://doi.org/10.18637/jss.v100.i055. **Vehtari, A., Gelman, A., Simpson, D., Carpenter, B., & Bürkner, P.-C. (2021)**. Rank-Normalization, Folding, and Localization: An Improved R-hat for Assessing Convergence of MCMC. *Bayesian Analysis, 16*(2), 667–718.---### KCI 등재 학술지6. **한국어 학습자의 쓰기 수행 평가 신뢰도 분석 — 다국면 라쉬 모형을 사용하여** (KCI 논문번호: ART002387352). KCI 포털(www.kci.go.kr)에서 전문 검색 가능.7. **학문 목적 한국어 말하기 평가 과제 유형 개발 연구** (KCI 논문번호: ART001912809). KCI 포털에서 검색 가능.8. **문항반응이론을 활용한 학문 목적 한국어 말하기 평가 문항 및 채점자 특성 분석** (KCI 논문번호: ART002419170). KCI 포털에서 검색 가능.---> **KCI 검색 방법**: www.kci.go.kr → 논문검색> 추천 검색어: `다국면 라쉬`, `MFPCM`, `문항 단계 난이도`, `한국어 평가`, `채점자 신뢰도`